> **The counterintuitive insight this technique is built on:**
>
> More memory is not always better. An agent that remembers everything eventually becomes slower to retrieve from, noisier in its responses, and unable to distinguish what matters now from what mattered two years ago. **Forgetting on purpose** — systematically pruning low-value, stale, and redundant memories — is what keeps a long-running memory store performant, relevant, and trustworthy.

---

## What Is Forgetting and Decay?

### The core idea in one sentence
Deliberately remove or suppress memories that are no longer useful — using **age**, **access patterns**, **importance scores**, or **budget constraints** — so the memory store stays lean, fast, and high-signal.

---

### The relationship to Technique 18

Technique 18 (Temporal Memory) used time-decay to **weight** memories in retrieval — old memories scored lower but were still stored. Technique 19 takes the next step: actually **removing or archiving** memories that have fallen below a usefulness threshold.

| | Temporal Memory (18) | Forgetting and Decay (19) |
|---|---|---|
| Old memories | Still stored, lower retrieval score | **Pruned** — removed from active store |
| Mechanism | Score weighting | Physical eviction / archival |
| Store size | Grows indefinitely | **Bounded** |
| Storage cost | Grows with time | Controlled |

---

### Why forgetting matters at scale

A FinCoach instance serving one user for three years might accumulate:
- 500+ conversation turns (vector store)
- 36 episode records (episodic store)
- 20+ semantic facts (semantic store)
- 15+ procedures (procedural store)

Most of those 500 conversation turns are irrelevant by year three. The market view from 2022 is wrong. The salary from the first session is three promotions out of date. The question about ELSS from 2023 has been answered 20 times since. Keeping all of this:
- Slows retrieval (more vectors to search)
- Increases noise (stale facts compete with current ones)
- Wastes storage
- May surface outdated advice as context

Forgetting on purpose fixes all four.

---

### Four forgetting strategies

**Strategy 1 — TTL (Time-To-Live)**
Every memory has a fixed expiration time. When it expires, it is deleted or archived.
- Pro: Simple, deterministic, predictable storage bounds
- Con: Rare-but-critical facts (user allergies, hard constraints) should NOT age out on a fixed schedule
- Best for: High-churn data (market views, session notes, search results)

**Strategy 2 — LRU (Least Recently Used)**
When storage is full, evict the memories that have not been accessed most recently. Each access resets the clock.
- Pro: Keeps what is actually being used; mirrors how OS caches work
- Con: Rare-but-critical facts may be pruned simply because they have not been retrieved lately
- Best for: High-frequency assistants where relevant context shifts quickly

**Strategy 3 — Importance-Weighted Eviction**
Assign an importance score to each memory: `importance = f(recency, access_count, semantic_relevance, category_weight)`. When pruning is needed, remove the lowest-importance memories first.
- Pro: Context-aware — keeps what matters, discards what does not
- Con: More complex; importance function requires tuning
- Best for: Long-running agents with diverse memory types
- Paper basis: arxiv 2604.02280 — "controlled pruning rather than heuristic removal"

**Strategy 4 — Budget-Constrained Pruning**
Set a hard limit on store size (N memories, B tokens, or S storage bytes). When the budget is exceeded, run the importance-weighted eviction to bring the store back within bounds.
- Pro: Guarantees bounded storage regardless of session length
- Con: Requires a budget decision at design time
- Best for: Production systems with cost/latency SLAs

---

### The critical failure mode: the rare-but-critical fact

Every forgetting strategy has the same failure mode: a low-frequency but high-importance fact gets pruned because it has not been used recently.

Examples: "User has a severe allergy to X." "User's investment constraint: never equity." "User is legally prohibited from trading in their employer's stock."

These facts may not have been retrieved in months — but when they matter, they are critical. The mitigation: **protection flags** on memory records. Protected memories are exempt from all eviction policies. In FinCoach, `category='constraint'` automatically sets `protected=True`.

---

### GDPR and right-to-be-forgotten

In regulated domains, forgetting is not just a performance feature — it is a legal requirement. GDPR Article 17 (right to erasure) requires that user data be deleted on request. A well-designed forgetting system makes this straightforward: the `delete_user(user_id)` function removes all memories for a user across all stores. Without this, GDPR compliance requires custom database surgery on every request.

---

## Trade-offs

| | |
|---|---|
| ✅ Bounded storage — cost controlled at scale | ❌ Risk of pruning rare-but-critical facts |
| ✅ Faster retrieval — smaller index to search | ❌ Importance scoring adds complexity |
| ✅ Less noise — stale memories cannot pollute retrieval | ❌ Pruned memories cannot be recovered (unless archived) |
| ✅ GDPR compliance — structured deletion is built-in | ❌ TTL bands require domain expertise to calibrate |
| ✅ Self-maintaining store — no manual curation | ❌ Protection flags must be set correctly at write time |

## Production Verdict

> **Essential for any agent running more than 30 days. Non-negotiable at scale.**
>
> Every production memory store eventually needs forgetting. The choice is not whether to forget — it is which strategy to use. Implement protection flags from day one (before you have data you regret losing) and combine TTL for high-churn data with importance-weighted eviction for the rest.

---

#### Some more information 

Controlled, purposeful information loss. Remembering everything is not a feature. It's a bug.

Think about a kitchen pantry. If you never throw anything out, it fills up with expired items. Finding fresh ingredients gets harder. Old cans push new ones to the back. A regular cleanup (checking dates, tossing what's stale) keeps the pantry useful.

It seems paradoxical that a memory system should deliberately forget. But unbounded memory accumulation creates real problems. As a memory store grows without pruning (removing low-value entries), retrieval quality degrades. The agent surfaces stale information alongside current facts. Embedding searches (finding entries by meaning similarity) return increasingly noisy results. Storage and inference costs climb. Worse, old memories can actively mislead. An outdated API endpoint or a superseded project plan sits in memory with the same retrieval score as the correction itself.

The insight that forgetting is adaptive comes from neuroscience. Richards and Frankland (2017) argued that the brain actively prunes memories unlikely to be useful. This frees capacity and reduces interference between similar memories. Hermann Ebbinghaus quantified this back in 1885 with his forgetting curve. He showed that retention decays exponentially without rehearsal but can be maintained through spaced repetition (reviewing at increasing intervals). These principles translate directly to agent memory systems.

In this notebook, you'll build a memory store with built-in decay and pruning. Each memory gets a strength score that decreases over time. When the agent retrieves a memory, its strength is reinforced. A background process periodically removes memories that fall below a threshold. The result is a self-regulating system: frequently useful memories persist, while irrelevant ones gracefully fade.

### Key Concepts

Ebbinghaus forgetting curve: The empirical finding that memory retention decays exponentially over time without reinforcement. The formula is R(t) = e^(-t/S) where S is memory stability. This provides the mathematical basis for all decay-based memory management.

Exponential decay function: The standard decay model: S(t) = S_0 * e^(-lambda * t). Here S_0 is the initial strength, lambda (the decay rate) controls how fast strength drops, and t is time since last access. The half-life (time to reach 50% strength) equals ln(2) / lambda.

Half-life: The time it takes for a memory's strength to drop to half its current value. A 24-hour half-life means an unaccessed memory loses half its strength every day. This is the main tuning knob for decay speed.

Spaced repetition: A reinforcement schedule where accessing a memory boosts its strength back up. Borrowed from learning science (the Leitner system, the SuperMemo SM-2 algorithm). In our system, each retrieval acts as a "review" that resets the decay clock.

Reinforcement boost: The amount of strength added when a memory is retrieved. A boost of 0.3 means a memory at strength 0.5 jumps to 0.8 after one retrieval. This rewards memories that prove useful.

Pruning threshold: A configurable minimum strength below which memories become candidates for removal. Memories below this value are either archived (soft delete, recoverable) or permanently removed (hard delete).

Storage pressure: When the memory store approaches capacity limits, the system can dynamically lower the pruning threshold or increase the decay rate. This triggers more aggressive forgetting to free space.

Cosine similarity: A measure of how similar two embedding vectors (lists of numbers encoding meaning) are. It ranges from -1 (opposite) to 1 (identical). We use it to find memories whose meaning is closest to a search query.

In [1]:
!pip install openai tiktoken --quiet

In [2]:
import json, os, time, math, uuid
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from enum import Enum

import openai
import tiktoken

In [3]:
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), "API key missing."

client    = openai.OpenAI(api_key=OPENAI_API_KEY)
CHAT_MODEL = "gpt-4o"
TOKENISER  = tiktoken.get_encoding("o200k_base")

print(f"Chat model: {CHAT_MODEL}")

Key loaded from environment variable.
Chat model: gpt-4o


---
## Part 1 — The Memory Record with Decay Metadata

In [ ]:
'''
What problem are we solving first?
Before we can "forget" anything, we need to answer one question: how do we decide WHICH memories to forget?
We need a way to look at any memory and say — this one is still useful, keep it. 
That one is old and nobody has asked about it in months, delete it. 
To make that decision, every memory needs to carry extra information about itself. 
That extra information is what this code sets up.

'''



'''
Dictionaries:
The first one — TTL_DAYS_BY_CATEGORY — is a table that says how long each TYPE of memory is allowed to live.
market_view   →   7 days     (market opinions go stale in a week)
session_note  →  14 days     (notes from one session, not needed long)
salary        → 180 days     (salary changes maybe once a year)
constraint    →  None        (NEVER expires — "never invest in equity" is forever)
Think of it like a best-before date on food. Market view memories expire in 7 days. 
Salary memories last 180 days. Some memories — constraints and personal identity — have NO expiry date and live forever.

The second one — CATEGORY_IMPORTANCE_WEIGHT — gives each type of memory a score from 0 to 1 saying 
how important it is to the agent. Constraints score 1.0 (maximum). Market views score 0.20 (minimum). 
This is used later when we have too many memories and need to decide which ones to delete first — 
the low-scoring ones go first.

'''

# TTL bands by category — how long each type of memory lives.
# None = never expires (protected categories).
TTL_DAYS_BY_CATEGORY: Dict[str, Optional[float]] = {
    "market_view":      7.0,    # Market sentiment — very short-lived
    "session_note":    14.0,    # Notes from a single session — medium short
    "investment_pref": 90.0,    # Preferences — moderate lifetime
    "salary":         180.0,    # Salary info — longer but does change
    "risk_profile":   365.0,    # Risk profile — long but not permanent
    "goal":           730.0,    # Long-term goals — very long
    "general":         60.0,    # Default
    "constraint":      None,    # Hard constraints — NEVER expire
    "personal":        None,    # Core identity facts — NEVER expire
}

# Category importance weights for the importance scoring formula.
# Higher = more important when deciding what to keep under budget pressure.
CATEGORY_IMPORTANCE_WEIGHT: Dict[str, float] = {
    "constraint":     1.0,   # Always maximum importance
    "personal":       0.95,
    "goal":           0.85,
    "risk_profile":   0.80,
    "salary":         0.75,
    "investment_pref":0.65,
    "general":        0.45,
    "session_note":   0.35,
    "market_view":    0.20,  # Lowest importance — expires fastest
}


'''
The DecayableMemory class — what it stores
This is just a data record. Think of it like a sticky note that a memory attaches to itself. Every memory carries:

content — what the memory actually says ("My salary is Rs 1,50,000")
category — what type of memory it is (salary, goal, constraint, etc.)
created_at — when was this memory written down? This never changes.
last_accessed — when was the last time someone actually USED this memory? This updates every time the memory is retrieved.
access_count — how many times has this memory been retrieved? Zero means nobody has looked at it.
protected — is this memory immune to deletion? True for constraints, True for personal facts. If protected is True, 
NO eviction strategy can touch it.
evicted — has this memory already been deleted? Once True, it no longer appears in searches.
eviction_reason — WHY was it deleted? Was it too old? Was it never used? This is the audit trail.


methods inside the class:
The four methods — this is where the interesting logic lives:

age_days() — simply answers: how many days old is this memory right now? Subtract creation time from today. 
A salary stored 200 days ago has age 200.

days_since_access() — how many days since anyone last retrieved this memory? Different from age. 
A memory can be 200 days old but accessed yesterday — days_since_access would be 1, not 200. 
This is used for LRU eviction (Least Recently Used).

is_expired_by_ttl() — checks: has this memory lived longer than its allowed TTL? 
If a market_view memory is 30 days old and its TTL is 7 days — yes, expired, delete it. 
If it is protected — no, never expires regardless of TTL.

importance_score() — this is the most important method. 
It answers: how valuable is this memory right now? It blends three things:

Recency (40% of the score): how fresh is it?  Fresh memories score high. Old memories score low.
Frequency (30% of the score): how often has it been used? A memory accessed 15 times is clearly useful. 
A memory accessed 0 times is suspicious.
Category weight (30% of the score): what type is it? 
A constraint is inherently more important than a session note regardless of age or usage.

The result is a number between 0 and 1. High score = keep. Low score = candidate for deletion.
record_access() — called every time a memory is retrieved. 
Updates last_accessed to today and increments access_count by 1. 
This is how the system knows the memory is still being used.

evict() — marks the memory as deleted. Sets evicted=True, records the reason and the timestamp. 
The memory is not physically destroyed — it moves to an archive. But it no longer appears in any retrieval results.


'''


@dataclass
class DecayableMemory:
    """
    A memory record with full decay and eviction metadata.
    Built on top of Technique 18's temporal awareness.
    Adds: TTL expiration, protection flags, importance scoring, eviction state.
    """

    memory_id:     str
    user_id:       str
    content:       str
    category:      str

    created_at:    str
    # ISO UTC — when stored. Immutable for audit trail.

    last_accessed: str
    # ISO UTC — when last retrieved. Updated on access.

    access_count:  int
    # How many times this memory has been retrieved.
    # High count = actively useful = protected from LRU eviction.

    protected:     bool
    # If True: NEVER evict, regardless of any policy.
    # Auto-set for category in ['constraint', 'personal'].
    # Can also be set manually for any critical memory.

    evicted:       bool = False
    # True once this memory has been removed from active storage.
    # Kept in the archive for audit; not returned in retrieval.

    eviction_reason: Optional[str] = None
    # Why this memory was evicted: 'ttl', 'lru', 'importance', 'budget', 'gdpr'.
    # Audit trail for eviction decisions.

    evicted_at:    Optional[str] = None
    # When it was evicted — for the decay log.

    session_id:    str = ""

    # ── Computed properties ──────────────────────────────────────────────

    def age_days(self) -> float:
        """Days since creation."""
        created = datetime.fromisoformat(self.created_at)
        now     = datetime.now(timezone.utc)
        if created.tzinfo is None:
            created = created.replace(tzinfo=timezone.utc)
        return (now - created).total_seconds() / 86400

    def days_since_access(self) -> float:
        """Days since last retrieved — used for LRU eviction."""
        accessed = datetime.fromisoformat(self.last_accessed)
        now      = datetime.now(timezone.utc)
        if accessed.tzinfo is None:
            accessed = accessed.replace(tzinfo=timezone.utc)
        return (now - accessed).total_seconds() / 86400

    def is_expired_by_ttl(self) -> bool:
        """
        Has this memory exceeded its TTL?
        Protected memories and categories with TTL=None never expire.
        """
        if self.protected:
            return False
            # Protected memories never expire — regardless of TTL.

        ttl = TTL_DAYS_BY_CATEGORY.get(self.category)
        if ttl is None:
            return False
            # No TTL defined = never expires.

        return self.age_days() > ttl

    def importance_score(
        self,
        recency_half_life_days: float = 30.0,
        recency_weight: float = 0.4,
        frequency_weight: float = 0.3,
        category_weight: float = 0.3,
        max_access_normaliser: int = 20,
    ) -> float:
        """
        Composite importance score from three signals.
        Based on arxiv 2604.02280 unified relevance formula:
          importance = w_r * recency + w_f * frequency + w_c * category_importance

        All weights must sum to 1.0. Returns a value in [0.0, 1.0].

        Args:
            recency_half_life_days: Half-life for the recency component.
            recency_weight:         Weight for time-based decay (w_r).
            frequency_weight:       Weight for access count (w_f).
            category_weight:        Weight for category importance (w_c).
            max_access_normaliser:  Access count to treat as maximum (score = 1.0).
        """
        if self.protected:
            return 1.0
            # Protected = always maximum importance.

        # Recency component: exponential decay from last access.
        # Uses LAST_ACCESSED, not created_at — recent retrieval refreshes importance.
        days_idle  = self.days_since_access()
        recency    = 0.5 ** (days_idle / recency_half_life_days)

        # Frequency component: normalised access count.
        frequency  = min(self.access_count / max_access_normaliser, 1.0)
        # Capped at 1.0 — 20+ accesses = maximum frequency score.

        # Category component: domain-specific importance weight.
        cat_score  = CATEGORY_IMPORTANCE_WEIGHT.get(self.category, 0.45)

        return (
            recency_weight   * recency   +
            frequency_weight * frequency +
            category_weight  * cat_score
        )

    def record_access(self) -> None:
        """Update access metadata on retrieval."""
        self.last_accessed = datetime.now(timezone.utc).isoformat()
        self.access_count += 1

    def evict(self, reason: str) -> None:
        """Mark this memory as evicted."""
        self.evicted        = True
        self.eviction_reason = reason
        self.evicted_at     = datetime.now(timezone.utc).isoformat()


print("DecayableMemory dataclass defined.")
print(f"TTL bands: {json.dumps({k: str(v)+' days' if v else 'never' for k,v in TTL_DAYS_BY_CATEGORY.items()}, indent=2)}")

DecayableMemory dataclass defined.
TTL bands: {
  "market_view": "7.0 days",
  "session_note": "14.0 days",
  "investment_pref": "90.0 days",
  "salary": "180.0 days",
  "risk_profile": "365.0 days",
  "goal": "730.0 days",
  "general": "60.0 days",
  "constraint": "never",
  "personal": "never"
}


---
## Part 2 — The ForgettingStore

The full memory store with four eviction strategies: TTL, LRU, importance-weighted, and budget-constrained.

In [ ]:
'''
What is the ForgettingStore?
Think of it as a filing cabinet with a janitor built in.
The filing cabinet stores memories. 
The janitor — on a schedule — walks through the cabinet and removes files that are no longer needed. 
The janitor has four different rules for deciding what to throw out. That is the ForgettingStore.
'''


class ForgettingStore:
    """
    An in-memory store that implements four forgetting strategies.

    In production: back this with ChromaDB or pgvector.
    The forgetting logic here is database-agnostic — it operates
    on the DecayableMemory objects, then syncs the deletes to the DB.
    
    When you create a ForgettingStore, you set three rules upfront:

    max_active_memories = 200 — the cabinet has space for 200 files maximum. 
    If it goes over, the janitor runs automatically.
    lru_evict_after_days = 60 — any memory not accessed in 60 days is a candidate for removal. 
    Like a library book that nobody has checked out in 2 months.
    importance_threshold = 0.25 — any memory scoring below 0.25 importance is a candidate for removal. 
    The scoring formula from Part 1 runs, and low scorers get evicted.
    
    There are also two storage areas:

    _active — memories currently available for retrieval. The agent can see these.
    _archive — memories that have been evicted. The agent cannot see these, but they are kept for compliance and audit. 
    In production these go to cold storage like S3.

    And one log:

    _eviction_log — a record of every single eviction: what was deleted, why, when, how old it was, 
    how many times it was accessed. This is your audit trail.
    
    """

    def __init__(
        self,
        user_id: str,
        max_active_memories: int = 200,
        # Hard budget: maximum number of active (non-evicted) memories.
        # When exceeded, importance-weighted eviction runs automatically.
        lru_evict_after_days: float = 60.0,
        # LRU: evict memories not accessed in this many days.
        # Protected memories are exempt.
        importance_threshold: float = 0.25,
        # Importance-weighted: evict memories scoring below this.
        # Default 0.25: aggressive pruning of low-value memories.
    ):
        self.user_id              = user_id
        self.max_active_memories  = max_active_memories
        self.lru_evict_after_days = lru_evict_after_days
        self.importance_threshold = importance_threshold

        self._active: Dict[str, DecayableMemory] = {}
        # Active memories — returned in retrieval.

        self._archive: Dict[str, DecayableMemory] = {}
        # Evicted memories — kept for audit, not returned in retrieval.
        # In production: move to cold storage (S3, BigQuery).

        self._eviction_log: List[Dict] = []
        # Audit trail of every eviction event.

        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[ForgettingStore] Initialised for user: {self.user_id}")
        print(f"  Budget       : {self.max_active_memories} memories max")
        print(f"  LRU threshold: {self.lru_evict_after_days} days idle")
        print(f"  Importance   : evict below {self.importance_threshold:.2f}")

    # ── WRITE ─────────────────────────────────────────────────────────────
    
    '''
    The add() method — writing a memory in
    This is straightforward. You give it the content, the category, and optionally a backdated timestamp (for simulating old memories in demos).
    The one interesting thing here: auto-protection. If the category is constraint or personal, 
    the memory is automatically marked protected=True. This happens silently — the developer does not have to remember to do it. If you store something with category constraint, it is immediately immune to all eviction strategies. Forever.
    
    '''

    def add(
        self,
        content: str,
        category: str,
        session_id: str = "",
        created_at: Optional[str] = None,
        force_protected: bool = False,
        # Force protection regardless of category.
    ) -> DecayableMemory:
        """Add a new memory. Auto-sets protection for critical categories."""

        now = datetime.now(timezone.utc).isoformat()
        ts  = created_at or now

        auto_protected = category in ("constraint", "personal") or force_protected
        # Constraints and personal identity facts are always protected.

        mem = DecayableMemory(
            memory_id    = str(uuid.uuid4()),
            user_id      = self.user_id,
            content      = content,
            category     = category,
            created_at   = ts,
            last_accessed = ts,
            access_count = 0,
            protected    = auto_protected,
            session_id   = session_id,
        )

        self._active[mem.memory_id] = mem

        age_label = f"(backdated {mem.age_days():.0f}d)" if created_at else "(now)"
        prot_label = " [PROTECTED]" if auto_protected else ""
        print(f"  [ADD] [{category:>16}] {age_label}{prot_label}: '{content[:55]}'")

        return mem

    # ── EVICTION STRATEGIES ───────────────────────────────────────────────
    
    '''
    Strategy 1 — run_ttl_eviction() — Time-To-Live
    The janitor looks at every memory and asks: has this memory lived longer than its allowed lifespan? It checks is_expired_by_ttl() from Part 1. If yes — archive it.

    A market_view memory older than 7 days gets archived.
    A salary memory older than 180 days gets archived.
    A constraint memory — never, because TTL is None.

    This is the cheapest strategy — just a timestamp comparison. No scoring needed.
    
    Strategy 2 — run_lru_eviction() — Least Recently Used
    The janitor looks at days_since_access for every memory. 
    If a memory has not been retrieved in more than lru_evict_after_days days — archive it.
    The key insight: this is about usage, not age. A memory can be 2 years old but if it was accessed yesterday, 
    LRU will not touch it. A memory can be 3 months old but if nobody has used it in 60 days, it gets archived.
    This mirrors how operating system caches work — hot files stay, cold files go.
    
    Strategy 3 — run_importance_eviction() — Importance-Weighted
    The janitor runs the importance score formula (from Part 1) on every memory. 
    Any memory scoring below the threshold — archive it.
    This is the most intelligent strategy because it combines all three signals at once: recency, 
    frequency, and category type. A memory that is old, rarely accessed, and low-category-importance gets a very low score and gets removed. A memory that is old but frequently accessed and high-category-importance stays.
    
    Strategy 4 — run_budget_eviction() — Budget-Constrained
    This one activates only when the cabinet is over its limit. If active count > max_active_memories:

    Calculate the overflow: how many memories need to be removed?
    Score all non-protected memories by importance.
    Remove the lowest-scoring ones first until we are back within budget.

    This is the safety net. Even if the other three strategies have been running regularly, 
    if something causes a spike in memory writes, this catches it.
    
    '''

    def run_ttl_eviction(self) -> int:
        """
        Strategy 1: TTL eviction.
        Remove all non-protected memories that have exceeded their TTL.
        Returns: count of evicted memories.
        """
        evicted = 0
        to_evict = [
            mem for mem in self._active.values()
            if not mem.protected and mem.is_expired_by_ttl()
        ]

        for mem in to_evict:
            self._do_evict(mem, reason="ttl")
            evicted += 1

        print(f"  [TTL] Evicted {evicted} expired memories | Active: {len(self._active)}")
        return evicted

    def run_lru_eviction(self) -> int:
        """
        Strategy 2: LRU eviction.
        Remove non-protected memories not accessed in lru_evict_after_days days.
        Returns: count of evicted memories.
        """
        evicted = 0
        to_evict = [
            mem for mem in self._active.values()
            if not mem.protected
            and mem.days_since_access() > self.lru_evict_after_days
        ]

        for mem in to_evict:
            self._do_evict(mem, reason="lru")
            evicted += 1

        print(f"  [LRU] Evicted {evicted} idle memories | Active: {len(self._active)}")
        return evicted

    def run_importance_eviction(
        self,
        threshold: Optional[float] = None,
    ) -> int:
        """
        Strategy 3: Importance-weighted eviction.
        Remove non-protected memories scoring below the importance threshold.
        Returns: count of evicted memories.
        """
        cutoff  = threshold if threshold is not None else self.importance_threshold
        evicted = 0

        to_evict = [
            mem for mem in self._active.values()
            if not mem.protected
            and mem.importance_score() < cutoff
        ]

        for mem in to_evict:
            self._do_evict(mem, reason=f"importance<{cutoff:.2f}")
            evicted += 1

        print(f"  [IMP] Evicted {evicted} low-importance memories (score<{cutoff}) | Active: {len(self._active)}")
        return evicted

    def run_budget_eviction(self) -> int:
        """
        Strategy 4: Budget-constrained eviction.
        If active count > max_active_memories:
          Evict the lowest-importance non-protected memories until within budget.
        Returns: count of evicted memories.
        """
        if len(self._active) <= self.max_active_memories:
            return 0  # Within budget — nothing to do.

        overflow = len(self._active) - self.max_active_memories
        # How many memories to remove.

        # Rank non-protected memories by importance, ascending (lowest first).
        candidates = sorted(
            [m for m in self._active.values() if not m.protected],
            key=lambda m: m.importance_score()
        )

        evicted = 0
        for mem in candidates[:overflow]:
            self._do_evict(mem, reason="budget_overflow")
            evicted += 1

        print(f"  [BUDGET] Over budget by {overflow} — evicted {evicted} | Active: {len(self._active)}")
        return evicted

    def run_all_strategies(self) -> Dict[str, int]:
        """
        Run all four strategies in the recommended order.
        Recommended order: TTL first (cheapest) → LRU → Importance → Budget.
        Each subsequent strategy catches what the previous ones missed.
        """
        print(f"\n[ForgettingStore] Running all eviction strategies...")
        print(f"  Before: {len(self._active)} active memories")
        results = {
            "ttl":        self.run_ttl_eviction(),
            "lru":        self.run_lru_eviction(),
            "importance": self.run_importance_eviction(),
            "budget":     self.run_budget_eviction(),
        }
        total = sum(results.values())
        print(f"  After:  {len(self._active)} active | {len(self._archive)} archived")
        print(f"  Total evicted this run: {total}")
        return results

    def _do_evict(self, mem: DecayableMemory, reason: str) -> None:
        """Internal: move a memory from active to archive."""
        mem.evict(reason)
        del self._active[mem.memory_id]
        self._archive[mem.memory_id] = mem

        self._eviction_log.append({
            "memory_id":       mem.memory_id,
            "category":        mem.category,
            "content_preview": mem.content[:50],
            "reason":          reason,
            "age_days":        round(mem.age_days(), 1),
            "access_count":    mem.access_count,
            "importance":      round(mem.importance_score(), 3),
            "evicted_at":      mem.evicted_at,
        })

    # ── GDPR: RIGHT TO ERASURE ────────────────────────────────────────────

    def delete_all_user_data(self) -> Dict[str, int]:
        """
        GDPR Article 17: Right to erasure.
        Permanently delete ALL memories for this user — active and archive.
        Returns count of records deleted.

        In production:
        - Cascade to all connected stores (vector DB, graph DB, entity store)
        - Log the deletion event for compliance audit
        - Return a deletion receipt with timestamp
        """
        active_count  = len(self._active)
        archive_count = len(self._archive)

        self._active.clear()
        self._archive.clear()
        self._eviction_log.clear()
        # All user data gone.

        deletion_receipt = {
            "user_id":        self.user_id,
            "deleted_at":     datetime.now(timezone.utc).isoformat(),
            "active_deleted": active_count,
            "archive_deleted":archive_count,
            "total_deleted":  active_count + archive_count,
        }

        print(f"[GDPR] All data deleted for user: {self.user_id}")
        print(f"  Active records deleted : {active_count}")
        print(f"  Archive records deleted: {archive_count}")

        return deletion_receipt

    # ── READ ──────────────────────────────────────────────────────────────

    def get_active(
        self,
        category_filter: Optional[List[str]] = None,
    ) -> List[DecayableMemory]:
        """Return all active (non-evicted) memories, optionally filtered by category."""
        active = list(self._active.values())
        if category_filter:
            active = [m for m in active if m.category in category_filter]
        return sorted(active, key=lambda m: m.importance_score(), reverse=True)

    # ── DIAGNOSTICS ───────────────────────────────────────────────────────

    def print_store_state(self, include_scores: bool = True) -> None:
        """Print full store state with decay scores."""
        print(f"\n{'='*72}")
        print(f"  ForgettingStore — User: {self.user_id}")
        print(f"  Active: {len(self._active)} | Archive: {len(self._archive)} | Budget: {self.max_active_memories}")
        print(f"{'='*72}")
        if include_scores:
            print(f"  {'Category':>16} | {'Age':>6} | {'Idle':>6} | {'Acc':>4} | {'Imp':>6} | {'TTLx':>5} | {'Prot':>5} | Content")
            print(f"  {'-'*16}-+-{'-'*6}-+-{'-'*6}-+-{'-'*4}-+-{'-'*6}-+-{'-'*5}-+-{'-'*5}-+---------")
            for mem in sorted(self._active.values(), key=lambda m: m.importance_score(), reverse=True):
                imp   = mem.importance_score()
                ttlx  = "YES" if mem.is_expired_by_ttl() else "no"
                prot  = "YES" if mem.protected else "no"
                print(f"  {mem.category:>16} | {mem.age_days():>5.0f}d | {mem.days_since_access():>5.0f}d | {mem.access_count:>4} | {imp:>6.3f} | {ttlx:>5} | {prot:>5} | {mem.content[:35]}...")
        print()
        if self._eviction_log:
            print(f"  Eviction log ({len(self._eviction_log)} events):")
            for ev in self._eviction_log[-5:]:
                print(f"    [{ev['reason']:>18}] [{ev['category']:>14}] age={ev['age_days']}d acc={ev['access_count']} imp={ev['importance']}: '{ev['content_preview']}'")
        print(f"{'='*72}\n")


print("ForgettingStore defined.")

ForgettingStore defined.


---
## Part 3 — Importance Score Demonstration

In [13]:
def demonstrate_importance_scoring() -> None:
    """Show importance scores for representative FinCoach memories."""

    # Get the current time in UTC — used as the reference point
    # to calculate how old each memory is and how long since it was accessed.
    now = datetime.now(timezone.utc)

    # This list defines 7 test scenarios — each represents a different
    # type of memory with different age, usage, and protection settings.
    # Format: (label, category, age_days, days_idle, access_count, protected)
    # - age_days   : how many days ago was this memory CREATED
    # - days_idle  : how many days since this memory was last ACCESSED
    # - access_count: how many times has this memory been retrieved
    # - protected  : True means this memory can NEVER be evicted
    scenarios = [
        # Salary stored just 3 days ago, accessed once, not protected
        ("New salary (fresh)",        "salary",          3,   3,  1, False),

        # Same type (salary) but stored 540 days ago (18 months), never accessed
        ("Old salary (18 months)",    "salary",        540, 540,  0, False),

        # Investment preference stored 90 days ago but accessed 8 times,
        # most recently 7 days ago — still actively being used
        ("Active SIP pref (weekly)",  "investment_pref", 90,  7,  8, False),

        # Market view from 60 days ago — never accessed, low importance type
        ("Old market view (2mo)",     "market_view",    60,  60,  0, False),

        # A hard constraint — "never invest in equity" — stored 180 days ago,
        # accessed twice, and marked protected=True so it can never be deleted
        ("Constraint (never equity)", "constraint",    180,  90,  2, True),

        # A session note from 30 days ago — never retrieved since it was written
        ("Old session note (1mo)",    "session_note",   30,  30,  0, False),

        # A retirement goal from 200 days ago — but accessed 15 times,
        # most recently just 10 days ago — clearly still in active use
        ("Freq-accessed goal",        "goal",          200,  10, 15, False),
    ]

    # Print the header lines for the output table
    print("IMPORTANCE SCORE BREAKDOWN")
    print("Formula: w_r*recency + w_f*frequency + w_c*category_weight")
    print(f"Weights: recency=0.4, frequency=0.3, category=0.3")
    print()

    # Print the column headers with fixed-width formatting
    # :>28 means right-align in a field of 28 characters
    print(f"{'Label':>28} | {'Recency':>8} | {'Freq':>6} | {'CatW':>6} | {'TOTAL':>7} | {'Keep?'}")
    print("-" * 75)

    # The threshold: any memory scoring below 0.25 is a candidate for eviction
    # This is configurable — in the ForgettingStore it defaults to 0.25
    THRESHOLD = 0.25

    # Loop through every scenario and score it
    for label, cat, age, idle, acc, prot in scenarios:

        # Convert age_days into an actual timestamp by going back 'age' days from now
        # e.g. age=540 → created 540 days ago
        created_ts  = (now - timedelta(days=age)).isoformat()

        # Convert days_idle into an actual timestamp for last_accessed
        # e.g. idle=7 → last accessed 7 days ago
        accessed_ts = (now - timedelta(days=idle)).isoformat()

        # Create a DecayableMemory object with all the metadata set above
        # This is the same object from Part 1 — we are just creating test instances
        mem = DecayableMemory(
            memory_id    = "x",           # Dummy ID — not relevant for scoring demo
            user_id      = "x",           # Dummy user
            content      = "x",           # Dummy content — not used in scoring
            category     = cat,           # The category determines TTL and category weight
            created_at   = created_ts,    # When was this memory created
            last_accessed = accessed_ts,  # When was this memory last retrieved
            access_count = acc,           # How many times has it been retrieved
            protected    = prot,          # Is this memory immune to eviction?
            session_id   = "x",           # Dummy session
        )

        # Calculate the full composite importance score using the formula:
        # importance = 0.4 * recency + 0.3 * frequency + 0.3 * category_weight
        # If protected=True, this always returns 1.0 — formula is bypassed
        imp = mem.importance_score()

        # ── Calculate each component separately for display purposes ──

        # RECENCY COMPONENT: exponential decay based on days_idle
        # Formula: 0.5 ^ (days_idle / 30)
        # Half-life here is 30 days — after 30 idle days, recency = 0.5
        # After 60 idle days, recency = 0.25. After 90 days, recency = 0.125
        # Note: uses days_idle (last accessed), NOT age (created_at)
        # A memory accessed yesterday scores high even if it is 2 years old
        recency_s = 0.5 ** (idle / 30)

        # FREQUENCY COMPONENT: normalised access count
        # Divide access_count by 20 (the maximum we care about)
        # min(..., 1.0) caps the result at 1.0 — 20 or more accesses = perfect score
        # 0 accesses = 0.0, 10 accesses = 0.5, 20+ accesses = 1.0
        freq_s = min(acc / 20, 1.0)

        # CATEGORY WEIGHT: look up this category's importance in the dictionary
        # constraint=1.0, personal=0.95, goal=0.85, salary=0.75, market_view=0.20
        # Default 0.45 if category not found in the dictionary
        cat_s = CATEGORY_IMPORTANCE_WEIGHT.get(cat, 0.45)

        # Decide KEEP or EVICT:
        # Keep if protected (regardless of score) OR if score >= threshold
        # Evict if not protected AND score < threshold
        keep = "KEEP" if (prot or imp >= THRESHOLD) else "EVICT"

        # Print one row of the results table
        # Each column is right-aligned to match the header widths
        print(f"{label:>28} | {recency_s:>8.3f} | {freq_s:>6.3f} | {cat_s:>6.3f} | {imp:>7.3f} | {keep}")

    # Print the footer explaining the threshold and protection rule
    print(f"\nEviction threshold: {THRESHOLD} — any memory scoring below this is a candidate")
    print("Protected memories always score 1.0 regardless of age or access.")


# Call the function — this runs the entire demonstration
demonstrate_importance_scoring()

IMPORTANCE SCORE BREAKDOWN
Formula: w_r*recency + w_f*frequency + w_c*category_weight
Weights: recency=0.4, frequency=0.3, category=0.3

                       Label |  Recency |   Freq |   CatW |   TOTAL | Keep?
---------------------------------------------------------------------------
          New salary (fresh) |    0.933 |  0.050 |  0.750 |   0.613 | KEEP
      Old salary (18 months) |    0.000 |  0.000 |  0.750 |   0.225 | EVICT
    Active SIP pref (weekly) |    0.851 |  0.400 |  0.650 |   0.655 | KEEP
       Old market view (2mo) |    0.250 |  0.000 |  0.200 |   0.160 | EVICT
   Constraint (never equity) |    0.125 |  0.100 |  1.000 |   1.000 | KEEP
      Old session note (1mo) |    0.500 |  0.000 |  0.350 |   0.305 | KEEP
          Freq-accessed goal |    0.794 |  0.750 |  0.850 |   0.797 | KEEP

Eviction threshold: 0.25 — any memory scoring below this is a candidate
Protected memories always score 1.0 regardless of age or access.


#### Output understanding

IMPORTANCE SCORE BREAKDOWN

Formula: w_r*recency + w_f*frequency + w_c*category_weight

Weights: recency=0.4, frequency=0.3, category=0.3

40% comes from how FRESH it is

30% comes from how OFTEN it has been used

30% comes from what TYPE of memory it is

All three together give you one final number between 0 and 1. That number decides keep or evict.

 Label |  Recency |   Freq |   CatW |   TOTAL | Keep?

Label — the name of the memory we are testing

Recency — freshness score (0 to 1)

Freq — how often it was accessed (0 to 1)

CatW — how important this TYPE of memory is (0 to 1)

TOTAL — the final combined score

Keep? — the final verdict

New salary (fresh)  |  0.933 | 0.050 | 0.750 | 0.613 | KEEP

Situation: Chiru's new salary of Rs 1,50,000 was stored just 3 days ago. He has retrieved it only once so far.

Recency = 0.933 → Nearly perfect. It was stored 3 days ago. Almost brand new. Very fresh.

Freq = 0.050 → Very low. Only accessed once. The agent has barely used this memory yet.

CatW = 0.750 → Salary is a fairly important type of memory so it gets a weight of 0.75.

Total = 0.613 → 0.4×0.933 + 0.3×0.050 + 0.3×0.750 = 0.373 + 0.015 + 0.225 = 0.613

Verdict = KEEP → Above the 0.25 threshold. Stays.

In plain English for students: Brand new salary. Barely used yet but very fresh. The freshness score alone is enough to keep it alive.


Old salary (18 months)  |  0.000 | 0.000 | 0.750 | 0.225 | EVICT

Situation: This is the OLD salary — Rs 90,000 — stored 540 days ago (18 months). It has never been accessed even once since it was stored.

Recency = 0.000 → 540 days old. The decay formula brings this to essentially zero. So old it has no freshness left.

Freq = 0.000 → Zero accesses. Nobody has ever retrieved this memory. Complete zero.

CatW = 0.750 → Same as above — salary is still an important category.

Total = 0.225 → 0.4×0 + 0.3×0 + 0.3×0.750 = 0 + 0 + 0.225 = 0.225

Verdict = EVICT → 0.225 is just below the 0.25 threshold. Gone.

In plain English for students: The old salary drags the total below 0.25 purely because it is ancient and nobody has ever 
asked about it. The category weight alone (0.225) is not enough to save it. The system correctly removes this stale, outdated fact.

Crucial point: This is the Rs 90,000 salary being thrown out while the Rs 1,50,000 new salary (Row 1) is kept. The system naturally prefers the fresh fact over the stale one.


Active SIP pref (weekly)  |  0.851 | 0.400 | 0.650 | 0.655 | KEEP

Situation: Chiru's SIP preference was stored 90 days ago but he has accessed it 8 times and most recently just 7 days ago.

Recency = 0.851 → Even though this memory is 90 days old, it was ACCESSED just 7 days ago. Recency is measured from last access, not creation. So it scores very high.
Freq = 0.400 → Accessed 8 times out of a maximum of 20. That gives 8/20 = 0.40.
CatW = 0.650 → Investment preferences are moderately important.
Total = 0.655 → Strong score. Well above 0.25.
Verdict = KEEP

In plain English for students: This memory is 3 months old but the agent keeps using it every week. The system sees that and says — clearly this memory is still valuable. Keep it. Age of creation does not matter. Age of last use is what matters.

Old market view (2mo)  |  0.250 | 0.000 | 0.200 | 0.160 | EVICT

Situation: "Markets are volatile — avoid investing right now." Stored 60 days ago. Never accessed once.

Recency = 0.250 → Market views have a half-life of 7 days. After 60 days it has gone through 8 half-lives. Recency is nearly dead at 0.250.
Freq = 0.000 → Never accessed. Zero.
CatW = 0.200 → Market views are the LOWEST importance category. They go stale fastest and matter least.
Total = 0.160 → 0.4×0.250 + 0.3×0.000 + 0.3×0.200 = 0.100 + 0 + 0.060 = 0.160
Verdict = EVICT

In plain English : Three strikes — old, never used, low importance type. This is a two-month-old opinion about market volatility. It is not just useless — it is actively dangerous because it might make the agent tell Chiru to avoid investing when markets have actually recovered. Correctly evicted.

Constraint (never equity)  |  0.125 | 0.100 | 1.000 | 1.000 | KEEP


Situation: "Never invest in equity under any circumstances." Stored 180 days ago. Accessed twice. Marked as protected.

Recency = 0.125 → 180 days old, last accessed 90 days ago. Naturally very low.
Freq = 0.100 → Only accessed twice out of 20. Low.
CatW = 1.000 → Constraints get the maximum category weight.

Now here is the twist. If you run the formula normally:

0.4×0.125 + 0.3×0.100 + 0.3×1.000 = 0.050 + 0.030 + 0.300 = 0.380
But the output shows 1.000, not 0.380.
Why? Because protected=True. The code checks — is this memory protected? Yes. So it returns 1.0 immediately and never even runs the formula.

Verdict = KEEP → Always. No matter what.

In plain English : This is the most important row. Even though this memory looks old and barely used, the protection flag means the maths never even runs. Score is locked at 1.0 forever. "Never invest in equity" must never be forgotten — not in 5 years, not in 10 years. The protection flag guarantees that.

Old session note (1mo)  |  0.500 | 0.000 | 0.350 | 0.305 | KEEP

Situation: A note from a session exactly 30 days ago. Never accessed since.

Recency = 0.500 → 30 days idle with a 30-day half-life. Exactly at the halfway point. Score = exactly 0.5.
Freq = 0.000 → Never accessed.
CatW = 0.350 → Session notes are low-importance.
Total = 0.305 → 0.4×0.500 + 0.3×0.000 + 0.3×0.350 = 0.200 + 0 + 0.105 = 0.305
Verdict = KEEP → But barely. Just above 0.25.

In plain English : This memory is right on the edge. It survived this time — but only just. If nobody accesses it for another 2 weeks, recency will drop further, the total will fall below 0.25, and it will be evicted in the next maintenance run. It is living on borrowed time.

Freq-accessed goal  |  0.794 | 0.750 | 0.850 | 0.797 | KEEP
Situation: "Goal: retire at 55, currently 32." Stored 200 days ago but accessed 15 times, most recently 10 days ago.

Recency = 0.794 → Accessed just 10 days ago → still high despite being 200 days old from creation.
Freq = 0.750 → 15 out of 20 accesses = 0.750. Very high.
CatW = 0.850 → Goals are near the top of the importance table.
Total = 0.797 → 0.4×0.794 + 0.3×0.750 + 0.3×0.850 = 0.318 + 0.225 + 0.255 = 0.797
Verdict = KEEP → Very strong score.

In plain English : A retirement goal that the agent keeps referencing — 15 times over 200 days. Frequency is rescuing what would otherwise be an ageing memory. The system correctly identifies this as one of the most valuable memories in the entire store.

Eviction threshold: 0.25 — any memory scoring below this is a candidate
Protected memories always score 1.0 regardless of age or access.
Two final rules printed as a reminder:
Rule 1: The cutoff is 0.25. Score below that — you are a candidate for eviction.
Rule 2: Protected memories skip the entire formula. They always score 1.0. The formula does not run for them


---
## Part 4 — Demo: All Four Strategies on a Real Store

In [7]:
print("FORGETTING DEMO — Simulating 3 years of FinCoach memory")
print("=" * 65)

store = ForgettingStore(
    user_id             = "chiru_001",
    max_active_memories = 15,    # Small budget for demo visibility.
    lru_evict_after_days = 30,   # LRU: evict after 30 idle days.
    importance_threshold = 0.25,
)

now = datetime.now(timezone.utc)

print("\nPopulating store with memories of various ages...")

memories_to_add = [
    # (content, category, age_days, access_count_simulation)
    ("My monthly salary is Rs 90,000",                       "salary",         540, 0),
    ("I work at Infosys as a software engineer",             "personal",        540, 3),
    ("Markets are volatile — avoid investing right now",     "market_view",      62, 0),
    ("Markets look stable — good time for debt SIPs",        "market_view",      45, 0),
    ("I prefer HDFC Short Duration Fund for debt exposure",  "investment_pref",  90, 4),
    ("My FD of Rs 50,000 is maturing in 3 months",           "general",         100, 0),
    ("Never invest in equity or stock markets for me",        "constraint",      180, 5),
    ("Discussed NPS and PPF comparison in this session",     "session_note",     35, 0),
    ("Discussed emergency fund building — target 6 months",  "session_note",     60, 0),
    ("Discussed retirement corpus target of Rs 2 crore",     "session_note",     90, 1),
    ("My monthly expenses are around Rs 60,000",             "salary",           90, 2),
    ("I changed jobs to TCS, new salary Rs 1,50,000",        "salary",            3, 0),
    ("Goal: retire at age 55, currently 32 years old",       "goal",            200, 6),
    ("Risk profile: conservative investor",                  "risk_profile",     180, 3),
    ("Considered liquid fund for emergency reserve",          "investment_pref",  95, 0),
    ("SIP in HDFC Liquid Fund started at Rs 5,000/month",   "investment_pref",  30, 2),
    ("Bond yields rising — potentially good for FD rates",   "market_view",       8, 0),
    ("Chiru asked about ELSS for 80C benefits",              "session_note",     120, 0),
    ("Interest rate discussion from last April",              "session_note",    425, 0),
]

for content, category, age, acc in memories_to_add:
    created_ts  = (now - timedelta(days=age)).isoformat()
    accessed_ts = (now - timedelta(days=max(1, age - acc * 5))).isoformat()
    # Simulate last_accessed = some days after creation, based on access count.

    mem = store.add(
        content    = content,
        category   = category,
        session_id = "demo",
        created_at = created_ts,
    )
    # Manually set access metadata for simulation.
    mem.last_accessed = accessed_ts
    mem.access_count  = acc

print(f"\nStore before forgetting: {len(store._active)} active memories")
store.print_store_state(include_scores=True)

print("\n" + "=" * 65)
print("Running all forgetting strategies...")
results = store.run_all_strategies()

print("\nStore AFTER forgetting:")
store.print_store_state(include_scores=False)

FORGETTING DEMO — Simulating 3 years of FinCoach memory
[ForgettingStore] Initialised for user: chiru_001
  Budget       : 15 memories max
  LRU threshold: 30 days idle
  Importance   : evict below 0.25

Populating store with memories of various ages...
  [ADD] [          salary] (backdated 540d): 'My monthly salary is Rs 90,000'
  [ADD] [        personal] (backdated 540d) [PROTECTED]: 'I work at Infosys as a software engineer'
  [ADD] [     market_view] (backdated 62d): 'Markets are volatile — avoid investing right now'
  [ADD] [     market_view] (backdated 45d): 'Markets look stable — good time for debt SIPs'
  [ADD] [ investment_pref] (backdated 90d): 'I prefer HDFC Short Duration Fund for debt exposure'
  [ADD] [         general] (backdated 100d): 'My FD of Rs 50,000 is maturing in 3 months'
  [ADD] [      constraint] (backdated 180d) [PROTECTED]: 'Never invest in equity or stock markets for me'
  [ADD] [    session_note] (backdated 35d): 'Discussed NPS and PPF comparison in this s

---
## Part 5 — The Critical Failure Mode: Rare-but-Important Fact

In [8]:
print("CRITICAL FAILURE MODE — The rare-but-important fact")
print("=" * 65)

risky_store = ForgettingStore(
    user_id              = "chiru_test",
    max_active_memories  = 5,
    lru_evict_after_days = 30,
    importance_threshold = 0.20,
)

now = datetime.now(timezone.utc)

# Add a critical constraint — but without protection (wrong category).
# This simulates what happens when the developer forgets to categorise correctly.
critical_but_unprotected = risky_store.add(
    content    = "CRITICAL: User has strict medical constraint — this maps to never invest in high-risk products",
    category   = "general",    # WRONG — should be 'constraint'
    created_at = (now - timedelta(days=200)).isoformat(),
)
critical_but_unprotected.access_count = 0       # Rarely accessed — not retrieved in months.
critical_but_unprotected.last_accessed = (now - timedelta(days=120)).isoformat()

# Add filler memories that are newer and more active.
for i in range(6):
    m = risky_store.add(
        content    = f"Recent advice note #{i+1} about standard financial planning",
        category   = "session_note",
        created_at = (now - timedelta(days=i*5)).isoformat(),
    )
    m.access_count = 2
    m.last_accessed = (now - timedelta(days=i*2)).isoformat()

print("\nBefore eviction:")
print(f"Critical constraint present: {critical_but_unprotected.memory_id in risky_store._active}")
print(f"Importance score           : {critical_but_unprotected.importance_score():.4f}")
print(f"Protected                  : {critical_but_unprotected.protected}")

risky_store.run_all_strategies()

print(f"\nAfter eviction:")
print(f"Critical constraint present: {critical_but_unprotected.memory_id in risky_store._active}")
print(f"Was evicted because        : {critical_but_unprotected.eviction_reason}")
print()
print("THE FAILURE: The critical constraint was evicted because:")
print("  1. category='general' — not protected")
print("  2. Old (200 days) and rarely accessed — low importance score")
print("  3. Budget pressure from newer memories — got pruned")
print()
print("THE FIX: Always use category='constraint' for critical constraints.")
print("  This auto-sets protected=True — prevents eviction by any strategy.")
print("  Or: explicitly pass force_protected=True when adding any critical memory.")

CRITICAL FAILURE MODE — The rare-but-important fact
[ForgettingStore] Initialised for user: chiru_test
  Budget       : 5 memories max
  LRU threshold: 30 days idle
  Importance   : evict below 0.20
  [ADD] [         general] (backdated 200d): 'CRITICAL: User has strict medical constraint — this map'
  [ADD] [    session_note] (backdated 0d): 'Recent advice note #1 about standard financial planning'
  [ADD] [    session_note] (backdated 5d): 'Recent advice note #2 about standard financial planning'
  [ADD] [    session_note] (backdated 10d): 'Recent advice note #3 about standard financial planning'
  [ADD] [    session_note] (backdated 15d): 'Recent advice note #4 about standard financial planning'
  [ADD] [    session_note] (backdated 20d): 'Recent advice note #5 about standard financial planning'
  [ADD] [    session_note] (backdated 25d): 'Recent advice note #6 about standard financial planning'

Before eviction:
Critical constraint present: True
Importance score           : 0.1600


---
## Part 6 — GDPR: Right to Erasure

In [9]:
print("GDPR DEMO — Right to Erasure (Article 17)")
print("=" * 65)

gdpr_store = ForgettingStore(
    user_id             = "chiru_gdpr_test",
    max_active_memories = 100,
)

for i in range(5):
    gdpr_store.add(
        content  = f"Financial fact #{i+1} for this user",
        category = "general",
    )
gdpr_store.add(
    content  = "Never invest in equity — constraint",
    category = "constraint",
)

print(f"\nBefore GDPR request:")
print(f"  Active memories : {len(gdpr_store._active)}")
print(f"  Archive memories: {len(gdpr_store._archive)}")

print("\nUser submits GDPR Article 17 Right to Erasure request...")
receipt = gdpr_store.delete_all_user_data()

print(f"\nAfter GDPR erasure:")
print(f"  Active memories : {len(gdpr_store._active)}")
print(f"  Archive memories: {len(gdpr_store._archive)}")
print(f"\nDeletion receipt:")
print(json.dumps(receipt, indent=2))

print()
print("Production GDPR implementation checklist:")
print("  1. Cascade to ALL stores: vector DB, graph DB, entity store, episodic store")
print("  2. Issue a deletion receipt with timestamp — compliance audit document")
print("  3. Complete within 30 days (GDPR requirement)")
print("  4. Log the request and completion — separate audit trail, not in user data")
print("  5. Even protected memories (constraints) must be deleted on GDPR request")
print("     (Protection only blocks automated eviction — not manual deletion)")

GDPR DEMO — Right to Erasure (Article 17)
[ForgettingStore] Initialised for user: chiru_gdpr_test
  Budget       : 100 memories max
  LRU threshold: 60.0 days idle
  Importance   : evict below 0.25
  [ADD] [         general] (now): 'Financial fact #1 for this user'
  [ADD] [         general] (now): 'Financial fact #2 for this user'
  [ADD] [         general] (now): 'Financial fact #3 for this user'
  [ADD] [         general] (now): 'Financial fact #4 for this user'
  [ADD] [         general] (now): 'Financial fact #5 for this user'
  [ADD] [      constraint] (now) [PROTECTED]: 'Never invest in equity — constraint'

Before GDPR request:
  Active memories : 6
  Archive memories: 0

User submits GDPR Article 17 Right to Erasure request...
[GDPR] All data deleted for user: chiru_gdpr_test
  Active records deleted : 6
  Archive records deleted: 0

After GDPR erasure:
  Active memories : 0
  Archive memories: 0

Deletion receipt:
{
  "user_id": "chiru_gdpr_test",
  "deleted_at": "2026-06-15T

---
## Part 7 — Forgetting Schedule: When to Run Each Strategy

In [10]:
print("PRODUCTION FORGETTING SCHEDULE")
print("=" * 65)

print("""
TRIGGER-BASED SCHEDULE (recommended for FinCoach):
─────────────────────────────────────────────────────────────────
Event                    | Strategy         | Why
─────────────────────────────────────────────────────────────────
After every session      | TTL check        | Fast, cheap, catches expired market views
Daily (2 AM cron)        | TTL + LRU        | Removes idle and expired memories
Weekly (Sunday)          | All strategies   | Full maintenance cycle
When store > 90% budget  | Budget eviction  | Reactive — triggered by size pressure
On GDPR request          | Full user delete  | Compliance — immediate, all stores


TTL BANDS FOR FINCOACH:
─────────────────────────────────────────────────────────────────
Market views         7 days   (rates, market sentiment — change weekly)
Session notes       14 days   (raw session reminders — superseded by episodes)
Investment prefs    90 days   (preferences drift but persist)
Salary             180 days   (changes infrequently)
Risk profile       365 days   (stable but may evolve)
Goals              730 days   (set for years)
Constraints         None      (permanent — never expire)
Personal identity   None      (permanent — name, age are durable)


PROTECTION POLICY:
─────────────────────────────────────────────────────────────────
Auto-protected   : category in ['constraint', 'personal']
Manual protection: force_protected=True on any critical memory
GDPR override    : delete_all_user_data() ignores protection flags
                   (user rights > system protection policy)


WHAT TO ARCHIVE VS DELETE:
─────────────────────────────────────────────────────────────────
Archive (cold storage):  session notes, old market views, old preferences
                          -- compliance audit may need them
Delete permanently:       on GDPR request
                          -- no exceptions
Never delete proactively: constraints, personal identity, compliance events
                          -- too risky to lose
""")

PRODUCTION FORGETTING SCHEDULE

TRIGGER-BASED SCHEDULE (recommended for FinCoach):
─────────────────────────────────────────────────────────────────
Event                    | Strategy         | Why
─────────────────────────────────────────────────────────────────
After every session      | TTL check        | Fast, cheap, catches expired market views
Daily (2 AM cron)        | TTL + LRU        | Removes idle and expired memories
Weekly (Sunday)          | All strategies   | Full maintenance cycle
When store > 90% budget  | Budget eviction  | Reactive — triggered by size pressure
On GDPR request          | Full user delete  | Compliance — immediate, all stores


TTL BANDS FOR FINCOACH:
─────────────────────────────────────────────────────────────────
Market views         7 days   (rates, market sentiment — change weekly)
Session notes       14 days   (raw session reminders — superseded by episodes)
Investment prefs    90 days   (preferences drift but persist)
Salary             180 days

---
## Part 8 — Live FinCoach Session with Forgetting-Aware Memory

In [11]:
# Minimal FinCoach agent using the ForgettingStore as the memory backend.

FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor for India.
Use the USER MEMORY CONTEXT provided to personalise your advice.
Prefer recent information over old. Respect stated constraints absolutely.
Keep responses to 3-5 sentences. Never recommend specific stocks."""


class FinCoachWithForgetting:
    """FinCoach backed by a ForgettingStore — memory decays and prunes automatically."""

    def __init__(self, user_id: str):
        self.user_id = user_id
        self.store   = ForgettingStore(
            user_id             = user_id,
            max_active_memories = 50,
            lru_evict_after_days = 45,
            importance_threshold = 0.20,
        )
        self._buffer: List[Dict] = []
        self._turn = 0

    def add_fact(
        self,
        content: str,
        category: str,
        created_at: Optional[str] = None,
    ) -> None:
        """Add a memory to the store."""
        self.store.add(content, category, created_at=created_at)

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """Send one message. Build context from the forgetting store."""

        # Build context from active memories (top by importance).
        active_mems = self.store.get_active()
        if active_mems:
            context_lines = [
                f"[{m.category} | imp={m.importance_score():.2f}] {m.content}"
                for m in active_mems[:8]  # Top 8 by importance.
            ]
            context = "USER MEMORY CONTEXT:\n" + "\n".join(context_lines)
        else:
            context = ""

        # Build messages.
        sys_content = FINCOACH_SYSTEM_PROMPT
        if context:
            sys_content += f"\n\n{context}"

        messages = [{"role": "system", "content": sys_content}]
        messages.extend(self._buffer[-6:])  # Last 3 turns.
        messages.append({"role": "user", "content": user_message})

        response = client.chat.completions.create(
            model=CHAT_MODEL,
            max_tokens=400,
            temperature=0.7,
            messages=messages,
        )
        reply = response.choices[0].message.content

        self._buffer.append({"role": "user",      "content": user_message})
        self._buffer.append({"role": "assistant", "content": reply})
        self._turn += 1

        # Store user message as a general memory.
        self.store.add(user_message, category="general")

        if verbose:
            print(f"[Turn {self._turn}] Active: {len(self.store._active)} memories | "
                  f"API: {response.usage.prompt_tokens} tokens")

        return reply

    def run_maintenance(self) -> None:
        """Run the forgetting cycle — call periodically in production."""
        print("\nRunning memory maintenance...")
        self.store.run_all_strategies()


# Build the agent and seed it with historical memories.
agent = FinCoachWithForgetting("chiru_001")
now   = datetime.now(timezone.utc)

# Seed memories at different ages.
agent.add_fact("Never invest in equity — hard constraint",       "constraint",      (now - timedelta(days=365)).isoformat())
agent.add_fact("Goal: retire at 55. Currently 32.",              "goal",            (now - timedelta(days=200)).isoformat())
agent.add_fact("Old salary was Rs 90,000 per month",             "salary",          (now - timedelta(days=540)).isoformat())
agent.add_fact("New salary at TCS: Rs 1,50,000 per month",       "salary",          (now - timedelta(days=5)).isoformat())
agent.add_fact("Markets were volatile last quarter",              "market_view",     (now - timedelta(days=100)).isoformat())

# Run maintenance before the session.
agent.run_maintenance()

print("\nLIVE SESSION with forgetting-aware memory")
print("=" * 65)

live_turns = [
    "What is my current salary and how much should I invest monthly?",
    "Given my goal of retiring at 55, am I on track?",
]

for turn in live_turns:
    print(f"\nUser: {turn}")
    response = agent.chat(turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

[ForgettingStore] Initialised for user: chiru_001
  Budget       : 50 memories max
  LRU threshold: 45 days idle
  Importance   : evict below 0.20
  [ADD] [      constraint] (backdated 365d) [PROTECTED]: 'Never invest in equity — hard constraint'
  [ADD] [            goal] (backdated 200d): 'Goal: retire at 55. Currently 32.'
  [ADD] [          salary] (backdated 540d): 'Old salary was Rs 90,000 per month'
  [ADD] [          salary] (backdated 5d): 'New salary at TCS: Rs 1,50,000 per month'
  [ADD] [     market_view] (backdated 100d): 'Markets were volatile last quarter'

Running memory maintenance...

[ForgettingStore] Running all eviction strategies...
  Before: 5 active memories
  [TTL] Evicted 2 expired memories | Active: 3
  [LRU] Evicted 1 idle memories | Active: 2
  [IMP] Evicted 0 low-importance memories (score<0.2) | Active: 2
  After:  2 active | 3 archived
  Total evicted this run: 3

LIVE SESSION with forgetting-aware memory

User: What is my current salary and how much sho

---
## Part 9 — The Complete Forgetting Decision Tree

In [12]:
print("THE FORGETTING DECISION TREE — Production Logic")
print("=" * 65)

print("""
When considering evicting a memory:

                    Is memory PROTECTED?
                    /                \\
                 YES                  NO
                  |                    |
             KEEP ALWAYS         Is TTL exceeded?
                                  /           \\
                               YES             NO
                                |               |
                            ARCHIVE        Is LRU threshold exceeded?
                         (cold storage)     /               \\
                                          YES                NO
                                           |                  |
                                       ARCHIVE      Is importance < threshold?
                                                     /                \\
                                                   YES                 NO
                                                    |                   |
                                                ARCHIVE         Is store over budget?
                                                                 /              \\
                                                               YES               NO
                                                                |                 |
                                                        Evict lowest        KEEP
                                                        importance
                                                        memories

SUMMARY:
  Protected      → Never evict (auto-set for 'constraint' and 'personal')
  TTL expired    → Archive (fast, cheap check — run after every session)
  LRU idle       → Archive (catches unused but not yet expired memories)
  Low importance → Archive (catches semantically low-value memories)
  Budget exceeded→ Evict lowest-importance to restore budget
""")

THE FORGETTING DECISION TREE — Production Logic

When considering evicting a memory:

                    Is memory PROTECTED?
                    /                \
                 YES                  NO
                  |                    |
             KEEP ALWAYS         Is TTL exceeded?
                                  /           \
                               YES             NO
                                |               |
                            ARCHIVE        Is LRU threshold exceeded?
                         (cold storage)     /               \
                                          YES                NO
                                           |                  |
                                       ARCHIVE      Is importance < threshold?
                                                     /                \
                                                   YES                 NO
                                                    |                

---
## Key Takeaways

**What you built:** A `DecayableMemory` dataclass with TTL expiration, LRU idle tracking, importance scoring (recency + frequency + category), and protection flags. A `ForgettingStore` with all four eviction strategies (TTL, LRU, importance-weighted, budget-constrained), a GDPR erasure function, an eviction log, and a live FinCoach demo.

---

### The three things to remember

1. **Protection flags must be set at write time — not at eviction time.** Once a critical memory is evicted it is gone from active retrieval. The protection mechanism only works if `force_protected=True` or `category='constraint'` is set when the memory is first stored. Retrofitting protection after months of data accumulation is painful. Build it in from day one.

2. **Run strategies in order: TTL first, then LRU, then importance, then budget.** TTL is the cheapest (no scoring needed, just compare a timestamp). LRU catches what TTL misses (old but within TTL, never accessed). Importance catches what LRU misses (recently accessed but semantically irrelevant). Budget catches everything else by forcing a hard cut. Each strategy is a safety net for the one before it.

3. **Importance scoring is the strategy that survives edge cases.** TTL fails for rare-but-critical facts. LRU fails for facts that matter but are not queried often. Importance scoring — weighting recency, access frequency, and category type — is the most robust eviction criterion because it captures all three dimensions of usefulness simultaneously.

---

